# Работа с данными

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [3]:
file_path = "../data/easy/raw/creditcard_2023.csv"
df = pd.read_csv(file_path)

In [4]:
print(f'Всего пропусков: {df.isnull().sum().sum()}')

Всего пропусков: 0


In [5]:
print(df.describe())

                  id            V1            V2            V3             V4  \
count  568630.000000  5.686300e+05  5.686300e+05  5.686300e+05  568630.000000   
mean   284314.500000 -5.118237e-17 -5.118237e-17  1.023647e-16       0.000000   
std    164149.486122  1.000001e+00  1.000001e+00  1.000001e+00       1.000001   
min         0.000000 -3.495584e+00 -4.996657e+01 -3.183760e+00      -4.951222   
25%    142157.250000 -5.652859e-01 -4.866777e-01 -6.492987e-01      -0.656020   
50%    284314.500000 -9.363846e-02 -1.358939e-01  3.528579e-04      -0.073762   
75%    426471.750000  8.326582e-01  3.435552e-01  6.285380e-01       0.707005   
max    568629.000000  2.229046e+00  4.361865e+00  1.412583e+01       3.201536   

                 V5            V6             V7            V8             V9  \
count  5.686300e+05  5.686300e+05  568630.000000  5.686300e+05  568630.000000   
mean   2.559118e-17  2.559118e-17       0.000000  1.279559e-17       0.000000   
std    1.000001e+00  1.0000

После df.describe() видно, что во всех колонках, кроме Amount, среднее 0, среднеквадратичное отклонение 1. Скейлим только ***Amount***.
Также датасет ***сбалансированный***: соотношение классов 50/50.

In [6]:
X = df.drop(columns=['id', 'Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [7]:
ct = ColumnTransformer([
    ('scaler', StandardScaler(), ['Amount'])
], remainder='passthrough')

ct.fit(X_train)  # X_train — DataFrame

# transform возвращает numpy, это нормально
X_train = ct.transform(X_train)
X_test = ct.transform(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

Разделили данные на train/test в соотношении 80/20, можем обучить **CatBoost**.
# Обучение CatBoost и предсказание

In [8]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    verbose=True,
    random_seed=42
)

In [9]:
model.fit(X_train, y_train)

0:	learn: 0.6290984	total: 204ms	remaining: 1m 41s
1:	learn: 0.5694399	total: 233ms	remaining: 57.9s
2:	learn: 0.5166229	total: 257ms	remaining: 42.6s
3:	learn: 0.4728176	total: 283ms	remaining: 35.1s
4:	learn: 0.4316325	total: 308ms	remaining: 30.5s
5:	learn: 0.3981496	total: 333ms	remaining: 27.4s
6:	learn: 0.3681184	total: 358ms	remaining: 25.2s
7:	learn: 0.3405945	total: 384ms	remaining: 23.6s
8:	learn: 0.3174152	total: 407ms	remaining: 22.2s
9:	learn: 0.2948570	total: 436ms	remaining: 21.3s
10:	learn: 0.2756101	total: 463ms	remaining: 20.6s
11:	learn: 0.2609359	total: 487ms	remaining: 19.8s
12:	learn: 0.2470371	total: 510ms	remaining: 19.1s
13:	learn: 0.2346519	total: 535ms	remaining: 18.6s
14:	learn: 0.2235100	total: 561ms	remaining: 18.1s
15:	learn: 0.2121819	total: 586ms	remaining: 17.7s
16:	learn: 0.2013480	total: 612ms	remaining: 17.4s
17:	learn: 0.1922081	total: 635ms	remaining: 17s
18:	learn: 0.1837465	total: 663ms	remaining: 16.8s
19:	learn: 0.1760019	total: 690ms	remainin

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.03, random_seed=42, verbose=True)

In [10]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [11]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

Accuracy: 0.9988
Precision: 0.9977
Recall: 1.0000
F1: 0.9988
ROC-AUC: 0.9999


# Вердикт
Датасет ***слишком хороший***. Данные идеально разделимы, CatBoost справился тут очень легко, да и подготавливать данные не пришлось.